## Conversión DICOM a NIFTI

In [ ]:
import os
import pandas as pd
import nibabel as nib
import dicom2nifti
import shutil
from pathlib import Path

# --- CONFIGURACION DE RUTAS ---
DICOM_ROOT = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI_original_dataset_downloaded/ADNI_MRI_1/ADNI"
BASE_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/workdir"

METADATA_CSV = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI_original_dataset_downloaded/ADNI_MRI_2_13_2026.csv"
COORDS_CSV = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI_original_dataset_downloaded/MAYOADIRL_MRI_MCH_12Feb2026.csv"

# Estructura de salida
pos_dir = os.path.join(BASE_DIR, "raw/positives") # MCH definitivos
neg_dir = os.path.join(BASE_DIR, "raw/negatives")

os.makedirs(pos_dir, exist_ok=True)
os.makedirs(neg_dir, exist_ok=True)

# --- CARGAR Y FILTRAR ---
df_meta = pd.read_csv(METADATA_CSV)
df_coords = pd.read_csv(COORDS_CSV)

# 1. Identificar Positivos Definitivos (MCH + Definite)
# MCH: Microhemorrhage (añadido por mi)
df_pos = df_coords[
    (df_coords['TYPE'] == 'MCH') & 
    (df_coords['STATUS'] == 'Definite') &
    (df_coords['RASLOCATIONS'].notna())
]
positive_ids = set(df_pos['LONI_IMG_ID'].astype(str).unique())

# 2. Identificar Negativos (Sanos)
# Usamos NOFINDINGS == 1 para asegurar que el radiologo confirmo que esta sano
negative_ids = set(df_coords[df_coords['NOFINDINGS'] == 1]['LONI_IMG_ID'].astype(str).unique())

print(f"IDs Positivos encontrados: {len(positive_ids)}")
print(f"IDs Negativos encontrados: {len(negative_ids)}")

# --- PROCESAMIENTO ---
processed_pos = 0
processed_neg = 0

for _, row in df_meta.iterrows():
    img_id = str(row['Image Data ID'])
    subject = row['Subject']
    
    # Determinar categoria
    target_folder = None
    if img_id in positive_ids:
        target_folder = pos_dir
    elif img_id in negative_ids:
        target_folder = neg_dir
    else:
        # Si es 'Possible' o no esta en la lista de sanos confirmados, lo ignoramos
        continue

    # Localizar carpeta DICOM
    subject_path = Path(DICOM_ROOT) / subject
    target_dicom_path = None
    
    if subject_path.exists():
        for p in subject_path.rglob(img_id):
            if p.is_dir():
                target_dicom_path = p
                break
    
    if target_dicom_path:
        dest_file = os.path.join(target_folder, f"{img_id}.nii.gz")
        
        if not os.path.exists(dest_file):
            print(f"Convirtiendo {img_id}...")
            try:
                # Reorientamos a RAS para que coincida con las coordenadas del CSV (añadido por mi)
                dicom2nifti.dicom_series_to_nifti(str(target_dicom_path), "temp.nii.gz", reorient_nifti=True)
                shutil.move("temp.nii.gz", dest_file)
                
                if target_folder == pos_dir:
                    processed_pos += 1
                else:
                    processed_neg += 1
            except Exception as e:
                print(f"Error procesando {img_id}: {e}")

print(f"Finalizado. Positivos: {processed_pos}, Negativos: {processed_neg}")

IDs Positivos encontrados: 1815
IDs Negativos encontrados: 4325
Convirtiendo I1591325...
Convirtiendo I1588330...
Convirtiendo I1553009...
Convirtiendo I1547741...
Convirtiendo I1045983...
Convirtiendo I1584289...
Convirtiendo I1333815...
Convirtiendo I1018178...
Convirtiendo I391677...
Convirtiendo I305919...
Convirtiendo I269118...
Convirtiendo I679910...
Convirtiendo I259780...
Convirtiendo I923854...
Convirtiendo I235793...
Convirtiendo I1605438...
Convirtiendo I958027...
Convirtiendo I1063918...
Convirtiendo I1029049...
Convirtiendo I1327454...
Convirtiendo I1622127...
Convirtiendo I1416084...
Convirtiendo I1175364...
Convirtiendo I1031691...
Convirtiendo I1603673...
Convirtiendo I983644...
Convirtiendo I1304807...
Convirtiendo I1168877...
Convirtiendo I401196...
Convirtiendo I323683...
Convirtiendo I245821...
Convirtiendo I215474...
Convirtiendo I1207296...
Convirtiendo I1229469...
Convirtiendo I1472514...
Convirtiendo I1170883...
Convirtiendo I326288...
Convirtiendo I277056...
C